# Batched calculation of auc roc

## 1. Motivation and framing the challenge

A main contribution of the project is being able to train an RL agent, such that it can learn the biasing 
dynamics of an acceptance loop and be therefore able to correct them.

To do so it needs to be able to compute some metrics on a subsample of the credit data at different stages
of the biasing process. Logically the longer a financial institution has existed and has performed applicants
selection, the larger the amount of data it will have.

For an off policy RL framework this means, that a replay buffer will need to handle features of shape
`[B, N, F]`, so batch size, sub sample of observations containing only "current information" - meaning
data from up to some round which has been observed, and some amount of features. Given that the data grows
with time, the amount of valid N entries will vary across batches. This makes that a normalized tensor of
shape `[B, N, F]` will *necessarily* contain unvalid positions.

Furthermore: for the RL framework to be feasibly trained it needs to be able to calculate the metric related
to the reward in a vectorized manner. Therefore the challenge is twofold:
 
1. Find a way to calculate in a vectorized manner `B` AUC-ROCs
2. Be able to find an adequate way to mask unvalid observations

## 2. Checking out the calculation of the AUC-ROC

Firs revist quickly and rather loosely how the ROC-Curve for binary classification is defined. The ROC-Curve
is a rank based method to assess the discriminative ability of a binary classifier. Without loss of generality
we can frame the binary classification problem as a regression problem, which tries to predict if a statistical
unit with observable features $x \in \mathcal{X}$, with $\mathcal{X}$ some feature space containing any kind
of features, is part of a class $y \in \{0, 1\}$ (note that $0, 1$ is an arbitrary encoding). Thereby the 
underlying assumption is that an observation belong either of the two classes - so there is no third option.
Then we can define a classifier $c : \mathcal{X} \to \mathcal{S} \subseteq \mathbb{R}$ as a function mapping 
some covariates $x \in \mathcal{X}$ to a scoring $s \in \mathcal{S}$, such that $c(x) = s$. The higher the scoring
the more likely it should be that a statistical unit belongs to class $1$.

Given a classifier $c$, the ROC curve is a plot of the true positive rate (TPR) on the vertical axis against 
the false positive rate (FPR) on the horizontal axis for every possible threshold setting. If the available data
was infinite the ROC-Curve would therefore be impossible to calculate, but given a finite sample, many beautiful
simplificatios can be done.